In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from multimodal_lancedb import MusicDatabase
from utils import *
from ground_truth import *
import pandas as pd
import lancedb

In [2]:
# Initialize the system
search_system = MusicSearchSystem(db_path="./.lancedb_2", music_dir="music")

In [3]:
db = lancedb.connect("./.lancedb_2")
table_audio = db.open_table("music_audio")
audio_embedding_df = table_audio.to_pandas()

audio_embedding_df

,song_name,song_path,audio_vector
0,My Rhapsody Sounds - Short Version A,music/Assaf Ayalon - My Rhapsody Sounds - Shor...,"[-0.029478367, 0.009709472, 0.05185532, 0.0254..."
1,Laid Back - Short Version A,music/The Mind Sweepers - Laid Back - Short Ve...,"[-0.033787563, 0.040226605, 0.004442984, 0.081..."
2,Far Taj,music/ZISO - Far Taj.mp3,"[-0.018721217, 0.035051364, 0.05141652, 0.0356..."
3,The Stones - Short Version,music/Wild Tulip - The Stones - Short Version.mp3,"[-0.008583562, 0.020294761, 0.0050604204, 0.03..."
4,Fixed - Short Version B,music/Swirling Ship - Fixed - Short Version B.mp3,"[0.0031058518, 0.020443501, 0.0060475464, -0.0..."
...,...,...,...
195,Orchestral News Intro,music/Tomasz_Redman - Orchestral News Intro.mp3,"[0.010141604, -0.017981885, 0.014272054, -0.03..."
196,Upbeat Happy Fun Logo,music/puremusic - Upbeat Happy Fun Logo.mp3,"[-0.031311695, -0.03104818, 0.022266045, 0.022..."
197,Happy Birthday In Paris,music/Music-Ideas - Happy Birthday In Paris.mp3,"[-0.049790498, -0.03626891, 0.059174698, -0.00..."
198,Funny Game Loop,music/honey_lemon - Funny Game Loop.wav,"[-0.045391183, -0.033096816, 0.025908915, -0.0..."


In [4]:
table_text = db.open_table("music_text")
text_embedding_df = table_text.to_pandas()

text_embedding_df

,source,song_name,artist,mood,video_theme,genre,instrument,other_tags,bpm,lmm_description,combined_info,text_vector
0,Artlist,My Rhapsody Sounds - Short Version A,Assaf Ayalon,"Uplifting, Happy, Carefree, Love, Playful","Business, Food, Education, Lifestyle, Urban","Cinematic, Acoustic, Pop, Folk, Children, Corp...","Acoustic Guitar, Keys",,145.0,A positive and uplifting acoustic folk track w...,"Moods: Uplifting, Happy, Carefree, Love, Playf...","[0.00016941165, -0.011131651, -0.004014101, -0..."
1,Artlist,Laid Back - Short Version A,The Mind Sweepers,"Powerful, Serious, Angry","Road Trip, Sport & Fitness, Fashion, Industry",Rock,"Electric, Guitar, Acoustic Drums",,78.0,This is a powerful and energetic rock music tr...,"Moods: Powerful, Serious, Angry. Video Themes:...","[-0.0041819224, -0.01964485, -0.021090291, -0...."
2,Artlist,Far Taj,ZISO,"Uplifting, Powerful, Carefree, Groovy","Travel, Shorts","World, Electronic, Hip Hop","Ethnic, Electronic Drums, Bass",,96.0,A traditional Indian Bhangra track with modern...,"Moods: Uplifting, Powerful, Carefree, Groovy. ...","[-0.011723319, -0.008885955, 0.0040757894, -0...."
3,Artlist,The Stones - Short Version,Wild Tulip,"Love, Serious, Dramatic, Sad, Hopeful","Time-Lapse, Documentary, Road Trip, Medical, L...",Cinematic,Piano,,69.0,This piece is a solo piano instrumental with a...,"Moods: Love, Serious, Dramatic, Sad, Hopeful. ...","[0.0012076573, -0.0034555339, -0.0020917628, -..."
4,Artlist,Fixed - Short Version B,Swirling Ship,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Ambient, Country, Cinematic","Electric Guitar, Synth, Electronic Drums, Pads",,121.0,"The music is mysterious and dramatic, featurin...","Moods: Serious, Dramatic, Scary, Dark. Video T...","[-0.0021377725, -0.012590199, -0.013713269, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...
195,envato,Orchestral News Intro,Tomasz_Redman,"energetic, epic, powerful, solemn, uplifting","announcement, broadcast news, broadcasting, bu...",corporate,strings,global,125.0,This is a dynamic and uplifting music track th...,"Moods: energetic, epic, powerful, solemn, upli...","[-0.006809455, -0.016746698, -0.016968248, -0...."
196,envato,Upbeat Happy Fun Logo,puremusic,"bouncy, bright, catchy, cheerful, energetic, f...","commercial, happy logo, intro, kids, logo, sum...","acoustic, children","claps, ukulele","melody, youth",NaN,"A positive, upbeat, cheerful, and happy acoust...","Moods: bouncy, bright, catchy, cheerful, energ...","[0.002132031, -0.005020746, -0.0029374287, -0...."
197,envato,Happy Birthday In Paris,Music-Ideas,"cheerful, funny, happy, lively, playful, upbeat","ads, advertising, birthday, broadway, casino, ...","bigband, jazz, retro","accordion, piano, trumpets","france, french, paris",120.0,A fun and lively Latin track featuring a varie...,"Moods: cheerful, funny, happy, lively, playful...","[-0.012678004, -0.011919039, -0.00089095806, -..."
198,envato,Funny Game Loop,honey_lemon,"comical, fun, funny, laugh, smile, soft","cartoon, comedy, comic, kids, short, summer, tv","acoustic, children, folk, jazz",bells,loop,170.0,"A casual, jazzy, swing music with vibraphone, ...","Moods: comical, fun, funny, laugh, smile, soft...","[-0.012116052, -0.01674075, 0.010771282, -0.02..."


In [53]:
# Search the music from query
query = '''I'm editing a fashion-themed video and looking for a groovy electronic/house track that captures the sleek energy of a runway or style showcase.'''
results = search_system.search_music(query, top_k=200)

# play music
# print("\nOverlapping Music：")
# for audio_path in results['audio_paths']:
#     print(f"\nNow playing: {os.path.basename(audio_path)}")
#     display(Audio(audio_path))

# Show explanation of LLM
df_recommendations = pd.DataFrame(results["final_results"])
df_recommendations

# print("\nLLM explanation：")
# print(results['explanation'])

[2025-05-19T04:02:15Z WARN  lance::dataset] No existing dataset at /home/tinglin/1125_env/experiment/.lancedb_temp/rerank_tmp.lance, it will be created


,song_name,artist,description,audio_distance,text_distance,similarity_score,similarity_audio,similarity_text,source,audio_path
0,Disco Guitar Groove,Frontmusic,"A groovy, funky, and upbeat disco track with a...",0.539318,0.130699,0.960213,0.950400,0.964419,"audio,text",music/Frontmusic - Disco Guitar Groove.mp3
1,Fashion House Loop,Sawtooz,A powerful and energetic house track with a ca...,0.593097,0.127196,0.957346,0.857820,1.000000,"audio,text",music/Sawtooz - Fashion House Loop.mp3
2,Positive Persistent Pluck Sequence 2,Artlist Musical Logos,A groovy and energetic royalty free electronic...,0.576188,0.131910,0.932558,0.886929,0.952114,"audio,text",music/Artlist Musical Logos - Positive Persist...
3,The Happy Intro,Korolkov,"A modern, upbeat and stylish electronic track ...",0.599668,0.135536,0.894653,0.846508,0.915287,"audio,text",music/Korolkov - The Happy Intro.mp3
4,Infinite - Short Version,Kuyani,A cool and groovy synthwave track with a retro...,0.584068,0.137833,0.886372,0.873363,0.891948,"audio,text",music/Kuyani - Infinite - Short Version.mp3
...,...,...,...,...,...,...,...,...,...,...
195,Variations on Ah vous dirai-je Maman - Var 3,Raviv Leibzirer,This is a fast and lively ragtime piece. It is...,0.859137,0.207064,0.252045,0.399830,0.188708,"audio,text",music/Raviv Leibzirer - Variations on Ah vous ...
196,Sad Piano,PineAppleMusic,"A subtle, delicate and thoughtful piano piece ...",0.700185,0.221578,0.230936,0.673467,0.041280,"audio,text",music/PineAppleMusic - Sad Piano.wav
197,Short Old England,R-Production,A traditional and elegant orchestral waltz wit...,0.665413,0.225642,0.219998,0.733328,0.000000,"audio,text",music/R-Production - Short Old England.mp3
198,Little Ragtime,DariusMusicProduction,"A vintage, retro, 1920's style ragtime piano t...",0.757549,0.219990,0.212598,0.574715,0.057404,"audio,text",music/DariusMusicProduction - Little Ragtime.mp3


In [54]:
df_recommendations_re = pd.DataFrame(results["final_rerank"])
df_recommendations_re

,song_name,artist,description,similarity_score,similarity_audio,similarity_text,source,rerank_score,audio_path
0,Fashion House Loop,Sawtooz,A powerful and energetic house track with a ca...,0.957346,0.857820,1.000000,"audio,text",0.996777,music/Sawtooz - Fashion House Loop.mp3
1,Disco Guitar Groove,Frontmusic,"A groovy, funky, and upbeat disco track with a...",0.960213,0.950400,0.964419,"audio,text",0.994488,music/Frontmusic - Disco Guitar Groove.mp3
2,Positive Persistent Pluck Sequence 2,Artlist Musical Logos,A groovy and energetic royalty free electronic...,0.932558,0.886929,0.952114,"audio,text",0.991748,music/Artlist Musical Logos - Positive Persist...
3,Infinite - Short Version,Kuyani,A cool and groovy synthwave track with a retro...,0.886372,0.873363,0.891948,"audio,text",0.938011,music/Kuyani - Infinite - Short Version.mp3
4,Star Dust - Short Version,Kuyani,"A cool and groovy, uplifting, and energetic, e...",0.868562,0.929095,0.842620,"audio,text",0.882022,music/Kuyani - Star Dust - Short Version.mp3
5,The Happy Intro,Korolkov,"A modern, upbeat and stylish electronic track ...",0.894653,0.846508,0.915287,"audio,text",0.824744,music/Korolkov - The Happy Intro.mp3
6,Energy Dance Loop,Difourks,A powerful and energetic electro track with du...,0.804046,0.584677,0.898061,"audio,text",0.799912,music/Difourks - Energy Dance Loop.wav
7,Energetic Loop,Artlist Musical Logos,"A hard-hitting, edgy, gritty, dark and driving...",0.816969,0.733742,0.852638,"audio,text",0.613237,music/Artlist Musical Logos - Energetic Loop.mp3
8,Event Ceremony Logo,Artlist Musical Logos,"This is a powerful, energetic and dynamic roya...",0.861065,0.873733,0.855636,"audio,text",0.348202,music/Artlist Musical Logos - Event Ceremony L...
9,Flow Free - Short Version,Manos Mars,"A dreamy, reflective, and introspective indie ...",0.844887,0.888936,0.826009,"audio,text",0.318485,music/Manos Mars - Flow Free - Short Version.mp3


In [59]:
top_k = 5

both = df_recommendations['song_name'].head(top_k).tolist()
audio_result = df_recommendations.sort_values("similarity_audio", ascending=False).head(top_k)['song_name'].tolist()
text_result = df_recommendations.sort_values("similarity_text", ascending=False).head(top_k)['song_name'].tolist()
rerank_result = df_recommendations_re['song_name'].head(top_k).tolist()

print(both)
print(audio_result)
print(text_result)
print(rerank_result)

['Disco Guitar Groove', 'Fashion House Loop', 'Positive Persistent Pluck Sequence 2', 'The Happy Intro', 'Infinite - Short Version']
['The Truth Is Close', 'Disco Guitar Groove', 'Action Opening', 'Star Dust - Short Version', 'Power Up']
['Fashion House Loop', 'Disco Guitar Groove', 'Positive Persistent Pluck Sequence 2', 'The Happy Intro', 'Energy Dance Loop']
['Fashion House Loop', 'Disco Guitar Groove', 'Positive Persistent Pluck Sequence 2', 'Infinite - Short Version', 'Star Dust - Short Version']


In [56]:
# db = lancedb.connect("./.lancedb_temp")
# rerank_tmp = db.open_table("rerank_tmp")
# rerank_tmp_df = rerank_tmp.to_pandas()

# rerank_tmp_df

In [60]:
evaluator = RankingEvaluator(top_k)

gt = Q20_GT
song_pool = df_recommendations["song_name"].tolist()
results = [evaluator.evaluate_random_baseline(gt, song_pool, n=10)] #Random 10 times

methods = ['Both(Audio+Text)', 'Audio', 'Text', 'Rerank']
recommendations = [both, audio_result, text_result, rerank_result]

for method, rec in zip(methods, recommendations):
    results.append(evaluator.evaluate(gt, rec, method))

df = pd.DataFrame(results)
df

,Method,Precision@5,Recall@5,nDCG@5,MAP
0,Baseline (Random),0.02,0.016667,0.014607,0.004167
1,Both(Audio+Text),0.60,0.500000,0.722727,0.500000
2,Audio,0.20,0.166667,0.213986,0.083333
3,Text,0.80,0.666667,0.853932,0.633333
4,Rerank,0.60,0.500000,0.722727,0.500000


In [23]:
def precision_at_k(gt, recs, k):
    recs_at_k = recs[:k]
    relevant = [r for r in recs_at_k if r in gt]
    return len(relevant) / k

def recall_at_k(gt, recs, k):
    recs_at_k = recs[:k]
    relevant = [r for r in recs_at_k if r in gt]
    return len(relevant) / len(gt) if len(gt) > 0 else 0.0

def ndcg_at_k(gt, recs, k):
    recs_at_k = recs[:k]
    dcg = 0.0
    for i, rec in enumerate(recs_at_k):
        if rec in gt:
            dcg += 1 / np.log2(i + 2)  # log2(i+2) since i starts from 0
    ideal_rels = [1] * min(len(gt), k)
    idcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels)])
    return dcg / idcg if idcg > 0 else 0.0

# MAP (Mean Average Precision)
def average_precision(gt, recs):
    hits = 0
    precisions = []
    for i, rec in enumerate(recs):
        if rec in gt:
            hits += 1
            precisions.append(hits / (i + 1))
    return sum(precisions) / len(gt) if gt else 0.0

In [24]:
import random

def evaluate_random_baseline_n_times(gt, pool, k=190, n=20):
    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    map_scores = []

    for _ in range(n):
        recs = random.sample(pool, k)
        precision_scores.append(precision_at_k(gt, recs, k))
        recall_scores.append(recall_at_k(gt, recs, k))
        ndcg_scores.append(ndcg_at_k(gt, recs, k))
        map_scores.append(average_precision(gt, recs))

    return {
        'Method': 'Baseline (Random)',
        'Precision@5': np.mean(precision_scores),
        'Recall@5': np.mean(recall_scores),
        'nDCG@5': np.mean(ndcg_scores),
        'MAP': np.mean(map_scores)
    }

In [55]:
import numpy as np

gt = [87, 91, 117, 124, 164]
b_recommended = [87, 5, 78, 62, 93]
a_recommended = [105, 5, 115, 84, 143]
t_recommended = [93, 78, 87, 76, 157]
rerank_recommended = [76, 78, 62, 93, 87]

song_pool = list(range(1, 201))
baseline_result = evaluate_random_baseline_n_times(gt, song_pool, k=5, n=10)
print(gt)

methods = ['B', 'A', 'T', 'RE']
recommendations = [b_recommended, a_recommended, t_recommended, rerank_recommended]

results = [baseline_result]
for method, rec in zip(methods, recommendations):
    results.append({
        'Method': method,
        'Precision@5': precision_at_k(gt, rec, 5),
        'Recall@5': recall_at_k(gt, rec, 5),
        'nDCG@5': ndcg_at_k(gt, rec, 5),
        'MAP': average_precision(gt, rec)
    })

df = pd.DataFrame(results)
df

[87, 91, 117, 124, 164]


,Method,Precision@5,Recall@5,nDCG@5,MAP
0,Baseline (Random),0.04,0.04,0.031565,0.011667
1,B,0.20,0.20,0.339160,0.200000
2,A,0.00,0.00,0.000000,0.000000
3,T,0.20,0.20,0.169580,0.066667
4,RE,0.20,0.20,0.131205,0.040000
